# Flower Classification — Full ML Pipeline

**ResNet50 · PyTorch · DVC · MLflow · Weights & Biases · GradCAM**

Oxford Flowers 102 — 102 categories · Best val accuracy: **96.96%**

> ⚠️ Enable GPU before running: Runtime → Change runtime type → T4 GPU

## 1 — Clone repository

In [ ]:
!git clone https://github.com/Katherina-B/ml-engineering-mlops.git
%cd ml-engineering-mlops
import sys
sys.path.insert(0, 'src')

Cloning into 'ml-engineering-mlops'...
remote: Enumerating objects: 50, done.
remote: Counting objects: 100% (50/50), done.
remote: Compressing objects: 100% (46/46), done.
remote: Total 50 (delta 23), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (50/50), 22.92 KiB | 7.64 MiB/s, done.
Resolving deltas: 100% (23/23), done.
/content/ml-engineering-mlops


## 2 — Install dependencies

In [ ]:
!pip install -q torch torchvision captum wandb mlflow dvc pyyaml tqdm python-dotenv

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 4.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.2/455.2 kB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.2/11.2 MB 72.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 103.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 76.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 470.1/470.1 kB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 265.9/265.9 kB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.3/79.3 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 451.2/451.2 kB 39.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

## 3 — Check GPU

In [ ]:
!python src/check_cuda.py

Cuda available: True


## 4 — Update config paths for Colab

In [ ]:
import yaml, os

base = '/content/ml-engineering-mlops'

with open('configs/params.yaml', 'r') as f:
    config = yaml.safe_load(f)

config['data']['local_dir']            = f'{base}/data/102flower'
config['artifacts']['output_dir']      = f'{base}/artifacts'
config['artifacts']['model_file']      = f'{base}/artifacts/model.pth'
config['artifacts']['best_model_file'] = f'{base}/artifacts/best_model.pth'
config['logging']['file']              = f'{base}/logs/training.log'

os.makedirs(f'{base}/artifacts', exist_ok=True)
os.makedirs(f'{base}/logs', exist_ok=True)

with open('configs/params.yaml', 'w') as f:
    yaml.dump(config, f)

print('Config updated ✓')

Config updated ✓


## 5 — Train (basic, no tracking)

In [ ]:
# Run from root so that configs/params.yaml is found correctly
!cd /content/ml-engineering-mlops && python src/train.py

100% 345M/345M [00:22<00:00, 15.6MB/s]
100% 502/502 [00:00<00:00, 2.39MB/s]
100% 15.0k/15.0k [00:00<00:00, 69.6MB/s]
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth
100% 97.8M/97.8M [00:00<00:00, 183MB/s]
Epoch 1: 100% 193/193 [01:48<00:00,  1.78it/s]
2026-08-14 19:34:45,972 - INFO - Epoch 1

## 6 — (Optional) MLflow tracking via DagsHub

Requires a [DagsHub](https://dagshub.com) account and access token.

In [ ]:
# Uncomment and fill in your credentials
import os
os.environ['MLFLOW_TRACKING_USERNAME'] = 'your_dagshub_username'
os.environ['MLFLOW_TRACKING_PASSWORD'] = 'your_dagshub_token'
os.environ['MLFLOW_TRACKING_PROJECTNAME'] = 'ml-engineering-mlops'
!cd /content/ml-engineering-mlops && python src/train_mlflow.py

MLflow tracking URI set to: https://dagshub.com/katherina.barbasheva/ml-engineering-mlops.mlflow
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Epoch 1: 100% 193/193 [01:46<00:00,  1.82it/s]
2026-08-14 21:11:32,262 - INFO - Epoch 1: Train Loss=1.9027, Train Accuracy=62.48%
2026-08-14 21:11:32,262 - INFO - Epoch 1: Val Loss=0.8501, Val Accuracy=79.12%
2026-08-14 21:11:40,932 - INFO - Epoch 1: Test Loss=0.8662, Test Accura

## 7 — (Optional) W&B experiment tracking

Requires a [Weights & Biases](https://wandb.ai) account.

In [ ]:
# Uncomment to enable W&B tracking
import wandb
wandb.login()  # will prompt for API key

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results


wandb: Enter your choice: 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.


wandb: Paste your API key and hit enter: ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: katherina-barbasheva (katherina-barbasheva-ntu-khpi) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## 8 — Interpretability: GradCAM + Saliency Maps

In [ ]:
%cd /content/ml-engineering-mlops

!git fetch origin
!git checkout origin/main -- src/train_mlflow.py


/content/ml-engineering-mlops
remote: Enumerating objects: 7, done.
remote: Counting objects: 100% (7/7), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 4 (delta 3), reused 0 (delta 0), pack-reused 0 (from 0)
Unpacking objects: 100% (4/4), 1.05 KiB | 1.05 MiB/s, done.
From https://github.com/Katherina-B/ml-engineering-mlops
   d39ddf7..5ea6ab0  main       -> origin/main


In [ ]:
!cd /content/ml-engineering-mlops && python src/interp.py

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: katherina-barbasheva (katherina-barbasheva-ntu-khpi) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Using an existing wandb-core service via WANDB_SERVICE.
wandb: ⢿ setting up run nm5j9jxp (0.0s)
wandb: ⣻ setting up run nm5j9jxp (0.0s)
wandb: ⣽ setting up run nm5j9jxp (0.0s)
wandb: Tracking run with wandb version 0.28.0
wandb: Run data is saved locally in /content/ml-engineering-mlops/wandb/run-20260814_201223-nm5j9jxp
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run gradcam_layers_lr_0.0001
wandb: ⭐️ View project at https://wandb.ai/katherina-barbasheva-ntu-khpi/flower-classification-interpretability
wandb: 🚀 View run at https://wandb.ai/katherina-barbasheva-ntu-khpi/flower-classification-interpretability/runs/nm5j9jxp
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pre

## 9 — Evaluate model

In [ ]:
!cd /content/ml-engineering-mlops && python src/evaluate.py

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Test Accuracy: 94.71%
Metrics saved to: /content/ml-engineering-mlops/artifacts/metrics.json


---
## Results

| Metric | Value |
|--------|-------|
| Best Validation Accuracy | **96.96%** (Epoch 19) |
| Best Validation Loss | 0.1296 |
| Architecture | ResNet50 (pretrained) |
| Dataset | Oxford Flowers 102 |